# 03. Demand Forecasting Model Development
## Time Series Analysis & Multi-Step Forecasting

### Objectives:
1. Aggregate appointment data into a continuous daily time series (498 calendar days).
2. Examine temporal properties, weekly seasonality (7-day cycle), and autocorrelation.
3. Apply strictly chronological train-test splits (no future data leakage).
4. Compare three distinct modeling approaches:
   - **Seasonal Naive / Moving Average Baseline**
   - **XGBoost Regressor** (Tree-based model with calendar, lag, and rolling features)
   - **SARIMAX Model** (Autoregressive integrated moving average with weekly seasonality)
5. Evaluate models across MAE, RMSE, MAPE (handling zero/low values safely), sMAPE, and R².
6. Export the fitted forecasting pipeline to `models/demand_forecast_model.joblib`.


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_raw_data, clean_dataset
from src.forecasting import (
    build_daily_time_series,
    engineer_forecasting_features,
    train_and_evaluate_forecasting,
    save_forecasting_artifact,
    forecast_future_demand
)

sns.set_theme(style='whitegrid')
%matplotlib inline


### 1. Build Daily Continuous Series & Chronological Split

In [ ]:
raw = load_raw_data('../data/raw.csv')
cleaned = clean_dataset(raw)
daily_all = build_daily_time_series(cleaned)

print(f"Daily series range: {daily_all['appointment_date'].min()} to {daily_all['appointment_date'].max()}")
print(f"Total days: {len(daily_all)}")
daily_all.head()


### 2. Autocorrelation & Seasonality Inspection

In [ ]:
plt.figure(figsize=(10, 4))
pd.plotting.autocorrelation_plot(daily_all['appointment_count'])
plt.title('Autocorrelation Plot of Daily Appointment Counts')
plt.xlim(0, 60)
plt.show()


### 3. Chronological Model Training & Evaluation

In [ ]:
results, best_model_name, best_model, test_eval_df = train_and_evaluate_forecasting(
    daily_all, test_ratio=0.20
)

# Tabulate forecasting comparison
results_df = pd.DataFrame(results).T[['mae', 'rmse', 'r2', 'smape', 'mape']]
results_df


### 4. Forecast Comparison on Chronological Test Set

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(test_eval_df['appointment_date'], test_eval_df['appointment_count'], label='Actual Appointments', color='#1f77b4', lw=2)
plt.plot(test_eval_df['appointment_date'], test_eval_df['XGBoost'], label='XGBoost Forecast', color='#ff7f0e', lw=1.8, ls='--')
plt.plot(test_eval_df['appointment_date'], test_eval_df['Seasonal_Naive'], label='Seasonal Naive (Lag 7)', color='#2ca02c', lw=1.2, ls=':')
plt.title('Daily Demand Forecast vs Actuals on Chronological Test Set')
plt.xlabel('Date')
plt.ylabel('Daily Appointments')
plt.legend()
plt.tight_layout()
plt.show()


### 5. Multi-Step Recursive Forecasting Demonstration

In [ ]:
artifact = {
    'model': best_model,
    'model_name': best_model_name,
    'metadata': {'metrics': results[best_model_name]}
}

future_14d = forecast_future_demand(artifact, daily_all, horizon_days=14)
print("Projected Next 14 Days Demand:")
future_14d[['appointment_date', 'day_of_week', 'predicted_appointments']]


### 6. Save Forecasting Artifact

In [ ]:
save_forecasting_artifact(
    model=best_model,
    model_name=best_model_name,
    metadata={
        'metrics': results[best_model_name],
        'all_models': results
    },
    filepath='../models/demand_forecast_model.joblib'
)
print("Forecasting model artifact saved successfully!")
